# Marching Cubes Variants

All three variants walk the same grid and share one interface; they
differ in how each cell's sign pattern is turned into triangles.
This page shows where the differences appear. The example meshes are
open surfaces, so they are rendered double-sided.


In [1]:
import torch

import isoext
from isoext import viewer
from isoext.utils import gaussian_smooth


## Watertightness

When two diagonally opposite corners of a cell face are inside and
the other two outside, the face admits two triangulations. The
`lorensen` tables, built with reflections, can pick incompatible
sides in neighboring cells, which leaves cracks. The `nagae` tables
avoid reflections and always produce closed meshes; so does
`lewiner`. Counting broken meshes over 50 random closed fields:


In [2]:
open_meshes = {"lorensen": 0, "nagae": 0, "lewiner": 0}
for seed in range(50):
    torch.manual_seed(seed)
    values = gaussian_smooth(torch.randn(12, 12, 12, device="cuda"), sigma=1.2)
    values[0], values[-1] = 1.0, 1.0
    values[:, 0], values[:, -1] = 1.0, 1.0
    values[:, :, 0], values[:, :, -1] = 1.0, 1.0
    grid = isoext.UniformGrid([12, 12, 12])
    grid.set_values(values)
    for method in open_meshes:
        v, f = isoext.marching_cubes(grid, method=method)
        edges = torch.cat([f[:, [0, 1]], f[:, [1, 2]], f[:, [2, 0]]]).sort(dim=-1).values
        _, counts = torch.unique(edges, dim=0, return_counts=True)
        open_meshes[method] += int((counts != 2).any())
print("fields with broken meshes:", open_meshes)


fields with broken meshes: {'lorensen': 49, 'nagae': 0, 'lewiner': 0}


## A Crack, Up Close

Two cells sharing an ambiguous face. `lorensen` triangulates the
shared face incompatibly on its two sides and leaves a slit:


In [3]:
values = torch.tensor(
    [0.1, 0.5, 0.4, 0.7, -0.6, 0.7, 1.0, -0.4, -1.0, -0.3, -0.2, -0.5],
    device="cuda",
).reshape(3, 2, 2)
pair = isoext.UniformGrid([3, 2, 2])
pair.set_values(values)

v, f = isoext.marching_cubes(pair, method="lorensen")
viewer.embed(v, f, color="steelblue", flat_shading=True, side="double", height=300)


The same field with `nagae` seals the crack:


In [4]:
v, f = isoext.marching_cubes(pair, method="nagae")
viewer.embed(v, f, color="seagreen", flat_shading=True, side="double", height=300)


## Topology of Ambiguous Cells

Closing the cracks still leaves a choice: are the pieces of surface
in an ambiguous cell connected or separate? Fixed tables always
answer the same way; `lewiner` evaluates the interpolant of the
corner values and follows it. In the cell below the fixed table
produces three separate pieces, while the interpolant connects two
of them:


In [5]:
values = torch.tensor(
    [0.5, -0.5, -0.2, 0.9, 0.2, 0.1, 0.3, -0.2], device="cuda"
).reshape(2, 2, 2)
cell = isoext.UniformGrid([2, 2, 2])
cell.set_values(values)

v, f = isoext.marching_cubes(cell, method="nagae")
print(f"nagae:   {f.shape[0]} triangles")
viewer.embed(v, f, color="seagreen", flat_shading=True, side="double", height=300)


nagae:   3 triangles


In [6]:
v, f = isoext.marching_cubes(cell, method="lewiner")
print(f"lewiner: {f.shape[0]} triangles")
viewer.embed(v, f, color="tomato", flat_shading=True, side="double", height=300)


lewiner: 5 triangles


### Lewiner Limitations

The `lewiner` variant ports the implementation of Lewiner et al.
(2003) and inherits its documented limitations: the interior test
examines the interpolant on a single section instead of analyzing it
fully, so it can misjudge tunnel connectivity in rare configurations,
and it is not invariant under reflections. The cell below shows the
second point: mirroring the values changes the extracted topology
from a tunnel into two separate sheets. Custodio et al. (2013)
document these issues and published corrections, which are not
included here; a corrected variant is planned. The meshes stay
watertight and crack free either way.


In [7]:
values = torch.tensor(
    [0.0625, -0.0625, 0.1875, 0.0625, 0.0625, 0.0625, -0.0625, -0.0625],
    device="cuda",
).reshape(2, 2, 2)
cell.set_values(values)
v, f = isoext.marching_cubes(cell, method="lewiner")
print(f"original: {f.shape[0]} triangles")
viewer.embed(v, f, color="tomato", flat_shading=True, side="double", height=300)


original: 9 triangles


In [8]:
cell.set_values(values.permute(2, 1, 0).contiguous())  # mirrored field
v, f = isoext.marching_cubes(cell, method="lewiner")
print(f"mirrored: {f.shape[0]} triangles")
viewer.embed(v, f, color="gold", flat_shading=True, side="double", height=300)


mirrored: 3 triangles


## Choosing a Variant

`lewiner` is the default: watertight and topologically faithful to
the sampled field, at about 10% more time than the fixed tables.
Use `nagae` when that margin matters and the topology of ambiguous
cells does not. `lorensen` is included for reference and comparison
with other implementations.


## Lookup Tables

The `lorensen` and `nagae` tables are generated by
[`luts/gen_mc_lut.py`](https://github.com/GuangyanCai/isoext/blob/master/luts/gen_mc_lut.py),
which reads base cases from a JSON file and expands them to all 256
configurations via rotations and, optionally, reflections. New
table-driven variants can be added the same way.

The `lewiner` tables are converted from the reference implementation
distributed with scikit-image (BSD) by
[`luts/convert_mc33_luts.py`](https://github.com/GuangyanCai/isoext/blob/master/luts/convert_mc33_luts.py),
rather than generated: MC33 tables couple triangulations with face
and interior test descriptors, and re-deriving that machinery is
where historical implementations accumulated bugs.


## References

- Lorensen and Cline, "Marching cubes: A high resolution 3D surface
  construction algorithm" (SIGGRAPH 1987)
- Nagae, Agui and Nagahashi, "Surface construction and contour
  generation from volume data" (1991)
- Chernyaev, "Marching Cubes 33: Construction of Topologically
  Correct Isosurfaces" (1995)
- Lewiner, Lopes, Vieira and Tavares, "Efficient implementation of
  Marching Cubes' cases with topological guarantees" (Journal of
  Graphics Tools, 2003)
- Custodio, Etiene, Pesco and Silva, "Practical considerations on
  Marching Cubes 33 topological correctness" (Computers & Graphics,
  2013)
